# Ensemble Results Processing
This code processes ensemble results from multiple model members for extreme event classification.

## Key Features:
* Ensemble Generation: Creates 20 model variants using different random seeds

* Multi-site Processing: Analyzes data for multiple geographical locations

* Dual Input Architecture: Combines local features (NN) and spatial data (CNN)

* Flexible Configuration: Supports both SPEI/SPI drought indices and soil moisture data

* Comprehensive Outputs: Generates both performance metrics and SHAP explainability results

## Output Files:
* Performance Metrics: Accuracy scores and prediction probabilities

* XAI Results: Ensemble-averaged SHAP values with standard deviations

* Text Reports: Detailed accuracy summaries for each

In [ ]:
# Library and module imports
import os
import sys
import torch
import hydra
from omegaconf import OmegaConf
from datetime import datetime
import numpy as np
import random
import pickle  # Added missing import
from pathlib import Path
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import balanced_accuracy_score
from quantifydrivers import machine_learning
import matplotlib.pyplot as plt
import yaml
import xarray as xr
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
from hydra import initialize, compose
from omegaconf import DictConfig, OmegaConf

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"*** Device set to: {device} ***")

from quantifydrivers.train_and_shap.config_schema import validate_schema
from quantifydrivers.train_and_shap.dataloading_script import build_datasets_and_loaders
from quantifydrivers.train_and_shap.training_script import training
from quantifydrivers.train_and_shap.evaluation_script import evaluation
from quantifydrivers.train_and_shap.SHAP_script import compute_SHAP

try:
    from hydra.core.global_hydra import GlobalHydra
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()
except Exception as e:
    print(f"Hydra Error: {e}")
with initialize(version_base=None, config_path=rel_config_path):
        cfg = compose(config_name="config",overrides=[])
    try:
        validated_cfg = validate_schema(cfg)
        print("Config Validation Passed!")
rel_config_path = "../train_and_shap/conf"

try:
    torch.use_deterministic_algorithms(True)
    print("Using deterministic algorithms.")
except Exception as e:
    print(f"Could not enforce deterministic algorithms: {e}")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

In [ ]:
def generate_ensemble_seeds(fixed_seed=123):
    rng = np.random.default_rng(fixed_seed)
    seeds = rng.integers(low=0, high=2**32 - 1, size=20).tolist()
    return seeds

def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # g.manual_seed(seed) # Passed explicitly later


list_seeds = generate_ensemble_seeds(fixed_seed=123)

sites = ["cordoba","hannover"]

device = torch.device("cpu") # Change to cuda if needed
print(f"Running on device: {device}")

for site in sites:

    # Initialize & validate config
    with initialize(version_base=None, config_path=rel_config_path):
        cfg = compose(config_name="config",overrides=[f"site={site}"])
    try:
        validated_cfg = validate_schema(cfg)
        print("Config Validation Passed!")

    except Exception as e:
        print("Config Validation Failed or validate_schema not imported.")
    # Create Generator and Seed everything
    g = torch.Generator()
    g.manual_seed(validated_cfg.seed)
    random.seed(validated_cfg.seed)
    np.random.seed(validated_cfg.seed)
    torch.manual_seed(validated_cfg.seed)

    datasets = build_datasets_and_loaders(configuration=validated_cfg, generator=g)
    test_dataset = datasets['test_dataset']

    _DATALOADERS_TEST_CONF = dict(batch_size= 32,drop_last=False,shuffle = False,num_workers=0)
    combined_test_loader = DataLoader(datasets['combined_test'], **_DATALOADERS_TEST_CONF)

    # Prepare for ensemble results
    outputs_prob_seeds = np.zeros((len(list_seeds), test_dataset.features.shape[0], 2))
    ensamble_shap_values_nn_raw = np.zeros((len(list_seeds), test_dataset.features.shape[0], test_dataset.features.shape[1]),dtype=np.float32)
    ensamble_shap_values_cnn_raw = np.zeros((len(list_seeds), test_dataset.features.shape[0], datasets['test_era5'].features.shape[1], 58, 124),dtype=np.float32)

    print(f"Ensemble arrays initialized. Output shape: {outputs_prob_seeds.shape}")

    for count_seed, seed in enumerate(list_seeds[:]):
        try:
            if validated_cfg.dataset.use_spei:
                file_res = os.path.join(validated_cfg.paths.results_dir, f'{site}', f'1lag_{distribution}_daily_{spei_spi}_{validated_cfg.percentile}_results_data_{seed}.pkl')
                file_shap = f'/gpfs/scratch/bsc32/bsc167965/data/test_train_n_shap_dilation/{site}/SHAP/1lag_{distribution}_daily_{spei_spi}_{validated_cfg.percentile}_SHAP_values_GradientExplainer_{seed}_{number_lags}lags.pkl'
            else:
                file_res = os.path.join(validated_cfg.paths.results_dir, f'{site}', f'{site}_{validated_cfg.percentile}_{seed}',f'{site}_{validated_cfg.percentile}_{seed}_evaluation.pkl')
                file_shap = os.path.join(validated_cfg.paths.results_dir, f'{site}', f'{site}_{validated_cfg.percentile}_{seed}',f'{site}_{validated_cfg.percentile}_{seed}_shap.pkl')

            with open(file_res, 'rb') as f:
                seed_results = pickle.load(f)

            with open(file_shap, 'rb') as f:
                shap_results = pickle.load(f)

        except FileNotFoundError as e:
            try:
                file_shap = os.path.join(validated_cfg.paths.results_dir, f'{site}', f'{site}_{validated_cfg.percentile}_{seed}',f'{site}_{validated_cfg.percentile}__{seed}_shap.pkl')
                with open(file_shap, 'rb') as f:
                    shap_results = pickle.load(f)
            except FileNotFoundError as e:
                print(f"    [!] File missing for seed {seed}: {e.filename}")
                continue

        outputs_prob_seeds[count_seed] = seed_results['out_probs_seed']
        ensamble_shap_values_nn_raw[count_seed] = shap_results['nn']
        ensamble_shap_values_cnn_raw[count_seed] = shap_results['cnn']

    print("Calculating Ensemble Statistics...")
    output_prob_ensamble = np.mean(outputs_prob_seeds,axis=0)
    std_ensamble = np.std(outputs_prob_seeds,axis=0)

    # Evaluate ensemble model -----------------------------------------------------------------------------------------------
    y_true, y_pred, extreme_acc, nonextreme_acc = machine_learning.evaluate_ensamble(test_loader=combined_test_loader, probs_ensamble=output_prob_ensamble, print_accuracies=True) # add extraction of output probabilities

    print("Calculating Mean Values...")
    # Mean  -----------------------------------------------------------------------------------------------------------------
    mean_ensamble_shap_values_nn_raw = np.mean(ensamble_shap_values_nn_raw,axis=0)
    mean_ensamble_shap_values_cnn_raw = np.mean(ensamble_shap_values_cnn_raw,axis=0)

    print("Calculating Standard Deviation...")
    # Standard deviation  ---------------------------------------------------------------------------------------------------
    ensamble_shap_values_nn_raw = ensamble_shap_values_nn_raw.astype(np.float32)
    ensamble_shap_values_cnn_raw = ensamble_shap_values_cnn_raw.astype(np.float32)
    std_ensamble_shap_values_nn_raw = np.std(ensamble_shap_values_nn_raw,axis=0)
    std_ensamble_shap_values_cnn_raw = np.std(ensamble_shap_values_cnn_raw,axis=0)

    print(f"Saving {site} Results...")
    # Dictionary to save results -----------------------------------------------------------------------------------------------
    site_results = {
            'y_true_pred_pairs': (y_true,y_pred),
            'out_probs_sites': output_prob_ensamble,
            'std_probs_sites': std_ensamble,
            'extreme_accuracy': extreme_acc,
            'nonextreme_accuracy': nonextreme_acc,
        }

    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    print("Preparing results text...")
    # Prepare results text
    results_text = (
        f"Site: {site}\n"
        f"Extreme Accuracy: {extreme_acc:.4f}\n"
        f"Non-Extreme Accuracy: {nonextreme_acc:.4f}\n"
        f"Balanced Accuracy: {balanced_acc:.4f}\n"
        "--------------------------------------\n"
    )

    # ============================================================================================================================
    # Save important results to txt file
    # ============================================================================================================================

    if validated_cfg.dataset.use_spei:
        with open(f"/gpfs/scratch/bsc32/bsc214253/results/txt/{validated_cfg.percentile}_results_data_{site}.pkl", "a") as f:  # "a" appends results for multiple sites
            f.write(results_text)

    else:
        save_dir = os.path.join(validated_cfg.paths.results_dir,"txt")
        os.makedirs(save_dir, exist_ok=True)
        pickle.dump(site_results, open(f"{save_dir}/{validated_cfg.percentile}_results_data_{site}.pkl", "wb"))

        file_path = f"{save_dir}/{validated_cfg.percentile}_results_data_{site}.txt"
        with open(file_path, "a") as f:
            f.write(results_text)

    print("Preparing shap results...")
    shap_results = {
            'nn_mean': mean_ensamble_shap_values_nn_raw, # These are from the current site
            'nn_std': std_ensamble_shap_values_nn_raw,
            'cnn_mean': mean_ensamble_shap_values_cnn_raw,
            'cnn_std': std_ensamble_shap_values_cnn_raw
        }
    if validated_cfg.dataset.use_spei:
        with open(f"/gpfs/scratch/bsc32/bsc214253/results/txt/{validated_cfg.percentile}_shap_data_{site}.pkl", "a") as f:  # "a" appends results for multiple sites
            f.write(shap_results)

    else:
        save_dir = os.path.join(validated_cfg.paths.results_dir,"txt")
        with open(os.path.join(save_dir, f"{save_dir}/{validated_cfg.percentile}_shap_data_{site}.pkl"), 'wb') as f:
                        pickle.dump(shap_results, f)

    print("RESULTS:")
    print(results_text)

print("SCRIPT COMPLETED SUCCESSFULLY")

## Loss plots ensambles

This block aggregates and visualizes the **training and validation loss curves** for the CombinedModel across multiple random seeds and sites.  

- **Input:** Loss histories (`losses_train`, `losses_val`) saved per seed and per site.  
- **Process:**  
  1. For each site, load loss curves for all seeds.  
  2. Trim sequences to the shortest run length to ensure alignment.  
  3. Compute mean and standard deviation across seeds for both training and validation.  
  4. Plot mean loss curves with shaded uncertainty bands.  
- **Outpu

In [ ]:
def generate_ensemble_seeds(fixed_seed=123):
    rng = np.random.default_rng(fixed_seed)  # Create reproducible random number generator
    seeds = rng.integers(low=0, high=2**32 - 1, size=20).tolist()  # Generate 20 random seeds
    return seeds

# Generate list of seeds (same every run because of fixed_seed)
list_seeds = generate_ensemble_seeds(fixed_seed=123)

# Sites to evaluate
sites = ['cordoba', 'hannover']

# Create subplot grid (2 rows x 3 columns) to plot training/validation losses per site
fig_site_loss, ax_site_loss = plt.subplots(2,3,figsize=(12,8))
ax_site_loss = ax_site_loss.flatten()  # Flatten for easy indexing

count_plot = 0  # Counter to track subplot index

# Loop over each site and aggregate results across seeds
for site in sites:

    losses_train_all_seeds = []  # Store training losses from all seeds
    losses_val_all_seeds = []    # Store validation losses from all seeds

    # Loop over seeds and load corresponding results
    for seed in list_seeds:

        # Path to the result files for this site and seed
        main_path = os.path.join(validated_cfg.paths.results_dir,f"{site}")
        file_losses = os.path.join(main_path, f'{site}_{validated_cfg.percentile}_{seed}',f'{site}_{validated_cfg.percentile}_{seed}_losses.pkl')
        with open(file_losses, 'rb') as f:
            seed_results = pickle.load(f)  # Load dictionary with loss histories

        # Extract training and validation loss curves
        loss_train = seed_results['losses_train']
        loss_val = seed_results['losses_val']

        losses_train_all_seeds.append(loss_train)
        losses_val_all_seeds.append(loss_val)

    # Ensure all loss sequences have the same length by trimming to shortest run
    min_len = min([len(loss) for loss in losses_train_all_seeds])
    train_losses_trimmed = np.array([loss[:min_len] for loss in losses_train_all_seeds])
    val_losses_trimmed = np.array([loss[:min_len] for loss in losses_val_all_seeds])

    # Compute mean and standard deviation across seeds for each epoch
    mean_train = train_losses_trimmed.mean(axis=0)
    std_train = train_losses_trimmed.std(axis=0)

    mean_val = val_losses_trimmed.mean(axis=0)
    std_val = val_losses_trimmed.std(axis=0)

    # Epoch indices (1-based)
    epochs = np.arange(1, min_len + 1)

    # Plot mean + uncertainty (std band) for training and validation losses
    ax_site_loss[count_plot].plot(epochs, mean_train, label='Train', color='blue')
    ax_site_loss[count_plot].fill_between(epochs, mean_train - std_train, mean_train + std_train, color='blue', alpha=0.2)
    ax_site_loss[count_plot].plot(epochs, mean_val, label='Val', color='orange')
    ax_site_loss[count_plot].fill_between(epochs, mean_val - std_val, mean_val + std_val, color='orange', alpha=0.2)

    # Add labels, title, legend, and axis limits
    ax_site_loss[count_plot].set_xlabel("epoch")
    ax_site_loss[count_plot].set_ylabel("loss")
    ax_site_loss[count_plot].set_title(f"CombinedModel Loss - {site}")
    ax_site_loss[count_plot].legend()
    ax_site_loss[count_plot].set_ylim(0.2, 0.8)  # Keep consistent y-limits for better comparison

    count_plot += 1  # Move to next subplot

# Output file name for the figure
name_save_losses_fig = "losses_all_locations_CombinedModel"

# Save figure in the specified folder
plot_save_path = os.path.join(config.paths.results_dir, "/plots/CombinedModel_losses")
plt.tight_layout()
plt.show()
os.makedirs(plot_save_path, exist_ok=True)  # Create directory if it does not exist
fig_site_loss.savefig(os.path.join(plot_save_path, f"95p_{name_save_losses_fig}.png"))
plt.close(fig_site_loss)  # Close figure to free memory
